# 数据类型与转换

学习目标：按数据含义选择列类型，转换文本数字并保留失败记录，判断缺失值、数值范围和数组转换带来的变化。

前置知识：数值与字符串、数组 dtype、缺失值、列选择、布尔条件与标签索引。

运行环境：Python 3.12、pandas 3.0、NumPy 2.5；Arrow 示例使用课程环境中的 PyArrow。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例使用单元内构造的小表，后续单元沿用导入的 pd 和 np。当前环境安装了 PyArrow，默认字符串存储后端为 pyarrow；本章不修改全局类型选项。

## 1 把文本数量转换为数值

数量以字符串保存时，不能直接当作数值计算。to_numeric 解析一列数字文本；errors="raise" 要求遇到无效文本时报告错误。

下面保留原始数量文本，在新列中保存解析结果，再计算每条记录增加一件后的数量。编号是标识，保留文本和前导零。

In [1]:
import pandas as pd
import numpy as np

orders = pd.DataFrame(
    {"code": ["001", "010", "020"], "quantity_text": ["3", "5", "8"]},
    index=["A", "B", "C"],
)
orders["quantity"] = pd.to_numeric(orders["quantity_text"], errors="raise")
orders["after_add"] = orders["quantity"] + 1

print(orders)
# 编号与原文本保留；quantity 为 3、5、8，after_add 为 4、6、9。
print(orders.dtypes)  # 两列文本为 str，两个数量列为 int64。
print(orders.index.tolist(), orders.shape)  # ['A', 'B', 'C']，(3, 4)。

  code quantity_text  quantity  after_add
A  001             3         3          4
B  010             5         5          6
C  020             8         8          9
code               str
quantity_text      str
quantity         int64
after_add        int64
dtype: object
['A', 'B', 'C'] (3, 4)


## 2 按目标类型转换

已经知道目标类型时，用 astype；DataFrame 可以按列名分别指定类型。它返回转换后的对象，原表不会因为这次调用而自动换类型。

下面把文本计数转换为 int64、文本单价转换为 float64。单价单位为元，这里用简单小数演示转换。

In [2]:
raw = pd.DataFrame(
    {"count": ["2", "4"], "price_yuan": ["1.5", "2.0"]}, index=["A", "B"]
)
typed = raw.astype({"count": "int64", "price_yuan": "float64"})

print(typed)
print(typed.dtypes)  # count 为 int64，price_yuan 为 float64。
print(raw.dtypes)  # 两列仍为 str。
print(typed.index.tolist(), typed.columns.tolist(), typed.shape)
# 行列标签及顺序不变，形状仍为 (2, 2)。

   count  price_yuan
A      2         1.5
B      4         2.0
count           int64
price_yuan    float64
dtype: object
count         str
price_yuan    str
dtype: object
['A', 'B'] ['count', 'price_yuan'] (2, 2)


### 2.1 转换失败不等于已清洗

astype 默认 errors="raise"，无效文本会报错。errors="ignore" 会在转换失败时保留原对象，不会把错误位置替换成缺失，也不能证明目标类型已经达到。

下面用一列输入观察失败；需要定位哪些值无法解析时，下一节采用 to_numeric。

In [3]:
raw = pd.Series(["3", "错误", "8"], index=["A", "B", "C"])
try:
    raw.astype("int64", errors="raise")
except ValueError as error:
    print(type(error).__name__)  # ValueError：错误文本无法转为整数。
else:
    raise AssertionError("预期无效文本无法转换为 int64")

unchanged = raw.astype("int64", errors="ignore")
print(unchanged.tolist(), unchanged.dtype)  # ['3', '错误', '8']，仍为 str。
print(raw.tolist())  # 原始值也仍保留。

ValueError
['3', '错误', '8'] str
['3', '错误', '8']


## 3 保留解析失败记录

to_numeric 的 errors="coerce" 将不能解析的内容变成缺失值。本来就缺失和解析失败都可能显示为缺失，因此要保留原始列，并用“原来非缺失、转换后缺失”定位本次失败。

下面约定空字符串也属于待核查的错误输入，原来的 None 则是已知缺失。notna 检查非缺失，isna 检查缺失；& 组合两个条件。

In [4]:
raw = pd.DataFrame(
    {"quantity_text": ["12", "错误", "", None]}, index=["A", "B", "C", "D"]
)
parsed = pd.to_numeric(raw["quantity_text"], errors="coerce")
failed = raw["quantity_text"].notna() & parsed.isna()
result = raw.copy()
result["quantity"] = parsed

print(result)
print(raw.loc[failed])  # 失败记录是 B、C；D 是输入原有缺失。
print(failed.tolist())  # [False, True, True, False]。
print(parsed.dtype, result.shape)  # float64，(4, 2)。

  quantity_text  quantity
A            12      12.0
B            错误       NaN
C                     NaN
D           NaN       NaN
  quantity_text
B            错误
C              
[False, True, True, False]
float64 (4, 2)


## 4 dtype 与缺失表示

dtype 描述列中的数据表示。pandas 既使用 NumPy 的 dtype，也使用扩展类型（ExtensionDtype）；扩展类型能提供 NumPy 基本类型没有的缺失表示或专用存储。

| dtype | 中文名称／含义 |
| --- | --- |
| int64 | NumPy 64 位有符号整数，本身不能保存 NaN |
| float64 | NumPy 64 位浮点数，可以用 NaN 表示缺失 |
| object | 保存 Python 对象，可混放不同类型，不等于文本专用类型 |
| str | pandas 3 默认字符串类型，用 NaN 表示缺失 |
| string | 显式 pandas 字符串类型，用 pd.NA 表示缺失 |
| Int64 | pandas 可空 64 位整数，大写 I 与 int64 不同 |
| Float64 | pandas 可空 64 位浮点数，用 pd.NA 表示缺失 |
| boolean | pandas 可空布尔类型，可保存 True、False、pd.NA |

下面同样包含两个整数和一个缺失位置，比较默认推断与显式可空整数。

In [5]:
inferred = pd.Series([1, None, 3])
nullable = pd.Series([1, None, 3], dtype="Int64")

print(inferred)  # float64：1.0、NaN、3.0。
print(nullable)  # Int64：1、<NA>、3。
print(inferred.isna().tolist(), nullable.isna().tolist())
# 缺失位置相同，都是 [False, True, False]；表示方式和 dtype 不同。
print(isinstance(inferred.dtype, np.dtype))  # True：NumPy dtype。
print(isinstance(nullable.dtype, pd.api.extensions.ExtensionDtype))  # True。

0    1.0
1    NaN
2    3.0
dtype: float64
0       1
1    <NA>
2       3
dtype: Int64
[False, True, False] [False, True, False]
True
True


## 5 object、str 与 string

object 不限定元素都是字符串；str 和 string 是字符串专用的扩展类型。pandas 3 从纯文本输入默认推断 str，缺失统一为 NaN；显式 dtype="string" 使用 pd.NA。

当前课程环境中，两种字符串类型都使用 PyArrow 存储。存储后端与缺失语义是两个维度，不能仅凭“底层都是 Arrow”就认为行为相同。StringDtype 的 storage 和 na_value 可以查看这两项约定。

In [6]:
objects = pd.Series(["甲", 2, None], dtype="object")
default_text = pd.Series(["甲", None, "乙"])
nullable_text = pd.Series(["甲", None, "乙"], dtype="string")

print(objects.tolist(), objects.dtype)  # ['甲', 2, None]，object。
print(default_text.tolist(), default_text.dtype)  # ['甲', nan, '乙']，str。
print(nullable_text.tolist(), nullable_text.dtype)  # ['甲', <NA>, '乙']，string。
print(default_text.dtype.storage, default_text.dtype.na_value)  # pyarrow，nan。
print(nullable_text.dtype.storage, nullable_text.dtype.na_value)  # pyarrow，<NA>。

['甲', 2, None] object
['甲', nan, '乙'] str
['甲', <NA>, '乙'] string
pyarrow nan
pyarrow <NA>


### 5.1 比较与字符串转换

两种字符串类型都可以用 isna 检查缺失，但比较结果不同：str 的缺失位置与普通文本作相等比较时得到 False；string 的对应位置保留 pd.NA，表示结果未知。

pandas 3 中 astype("str") 会保留缺失，不会把缺失变成字符串 "nan"。下面同时观察比较结果和实际转换。

In [7]:
default_text = pd.Series(["甲", None, "乙"], dtype="str")
nullable_text = pd.Series(["甲", None, "乙"], dtype="string")
print(default_text == "甲")  # [True, False, False]，bool。
print(nullable_text == "甲")  # [True, <NA>, False]；当前 Arrow 存储下为 bool[pyarrow]。
print(default_text.isna().tolist(), nullable_text.isna().tolist())
# 两边缺失位置都是 [False, True, False]。

numbers = pd.Series([1.5, None])
as_text = numbers.astype("str")
print(as_text.tolist(), as_text.dtype)  # ['1.5', nan]，str。
print(as_text.isna().tolist())  # [False, True]，缺失仍可被检测。

0     True
1    False
2    False
dtype: bool
0     True
1     <NA>
2    False
dtype: bool[pyarrow]
[False, True, False] [False, True, False]
['1.5', nan] str
[False, True]


### 5.2 字符串列不接收任意对象

给 str 列赋入数值会报 TypeError。若一列确实需要混放 Python 对象，应明确选用 object；若它表示文本，就应先决定数值该如何写成文本。不要依赖赋值时悄悄改变整个列的类型。

In [8]:
labels = pd.Series(["甲", "乙"], index=["A", "B"], dtype="str")
try:
    labels.loc["B"] = 2
except TypeError as error:
    print(type(error).__name__)  # TypeError：字符串列不能直接写入整数。
else:
    raise AssertionError("预期 str 列拒绝整数赋值")

mixed = labels.astype("object")
mixed.loc["B"] = 2
print(mixed.tolist(), mixed.dtype)  # ['甲', 2]，object。
print(labels.tolist(), labels.dtype)  # ['甲', '乙']，原字符串列不变。

TypeError
['甲', 2] object
['甲', '乙'] str


## 6 可空数值与布尔

整数计数存在缺失时可用 Int64；包含小数时可用 Float64；布尔判断存在未知状态时可用 boolean。下面每列都保留中间的缺失位置。

可空整数参与计算时，缺失通常传播到结果中；除法产生小数时，结果也不必保持整数类型。

In [9]:
readings = pd.DataFrame({
    "count": pd.Series([2, None, 6], dtype="Int64"),
    "value": pd.Series([1.5, None, 3.5], dtype="Float64"),
    "valid": pd.Series([True, None, False], dtype="boolean"),
})
print(readings)
print(readings.dtypes)  # Int64、Float64、boolean。
print(readings["count"] + 1)  # [3, <NA>, 7]，Int64。
print(readings["count"] / 2)  # [1.0, <NA>, 3.0]，Float64。
print(readings.isna())  # 三列的第 1 行都为 True。

   count  value  valid
0      2    1.5   True
1   <NA>   <NA>   <NA>
2      6    3.5  False
count      Int64
value    Float64
valid    boolean
dtype: object
0       3
1    <NA>
2       7
Name: count, dtype: Int64
0     1.0
1    <NA>
2     3.0
Name: count, dtype: Float64
   count  value  valid
0  False  False  False
1   True   True   True
2  False  False  False


### 6.1 pd.NA 表示未知

pd.NA 的比较结果通常仍为 pd.NA，不能用“等于 pd.NA”查找缺失。isna 给出明确的缺失标记。

单个 pd.NA 也不能直接解释为 Python 的 True 或 False；bool(pd.NA) 会报错。后面的可空布尔运算会按是否能确定结果处理未知值。

In [10]:
counts = pd.Series([2, None, 6], dtype="Int64")
print(counts > 3)  # [False, <NA>, True]，缺失位置无法判断是否大于 3。
print(counts == pd.NA)  # 三个位置都是 <NA>，不能用来定位缺失。
print(counts.isna().tolist())  # [False, True, False]。

try:
    bool(pd.NA)
except TypeError as error:
    print(type(error).__name__)  # TypeError：未知状态不能直接当作单个真假值。
else:
    raise AssertionError("预期 pd.NA 的布尔值不明确")

0    False
1     <NA>
2     True
dtype: boolean
0    <NA>
1    <NA>
2    <NA>
dtype: boolean
[False, True, False]
TypeError


## 7 可空布尔的三值逻辑

boolean 的逐元素逻辑运算包括 &（与）、|（或）、^（异或）。它们使用 True、False、pd.NA 三种状态：只有已知输入不足以确定结果时，结果才保留未知。

例如 False 与未知做“与”仍为 False，True 与未知做“或”仍为 True；异或需要知道双方是否不同，一方未知时不能确定。下面只列不重复的输入组合，交换左右输入不改变结果。

In [11]:
left = pd.Series([True, True, True, False, False, pd.NA], dtype="boolean")
right = pd.Series([True, False, pd.NA, False, pd.NA, pd.NA], dtype="boolean")
truth = pd.DataFrame({
    "left": left, "right": right,
    "AND": left & right, "OR": left | right, "XOR": left ^ right,
})

print(truth)
# True 与 NA：AND 为 NA，OR 为 True，XOR 为 NA。
# False 与 NA：AND 为 False，OR 为 NA，XOR 为 NA。
# 两边都为 NA 时，三个结果都为 NA。
print(truth.shape)  # (6, 5)，每行是一组逻辑输入及其结果。

    left  right    AND     OR    XOR
0   True   True   True   True  False
1   True  False  False   True   True
2   True   <NA>   <NA>   True   <NA>
3  False  False  False  False  False
4  False   <NA>  False   <NA>   <NA>
5   <NA>   <NA>   <NA>   <NA>   <NA>
(6, 5)


### 7.1 未知条件参与筛选

用 boolean 掩码筛选时，pd.NA 按未选中处理。若任务要求同时保留未知记录，先明确这个业务规则，再用 fillna(True) 构造筛选条件。

下面掩码与数据的标签完全一致；填充只是选择规则，不会把原来的未知判断改成已经确认。

In [12]:
rows = pd.DataFrame({"quantity": [3, 5, 8]}, index=["A", "B", "C"])
mask = pd.Series([True, pd.NA, False], index=rows.index, dtype="boolean")

confirmed = rows.loc[mask]
including_unknown = rows.loc[mask.fillna(True)]
print(confirmed)  # 仅 A，形状为 (1, 1)。
print(including_unknown)  # A、B，形状为 (2, 1)，保留原顺序。
print(confirmed.shape, including_unknown.shape)
print(mask.tolist())  # [True, <NA>, False]，原判断没有被改写。

   quantity
A         3
   quantity
A         3
B         5
(1, 1) (2, 1)
[True, <NA>, False]


## 8 转换前检查数值范围

固定宽度整数只能表示有限范围。np.iinfo 可以查看 NumPy 整数类型的最小值和最大值；选择较小类型前要先检查所有有效值是否落在范围内。

下面拟将计数保存成有符号 int8。先检查范围，再决定是否转换；不把强制转换当作范围检查。

In [13]:
counts = pd.Series([0, 127, 128], dtype="int64")
limits = np.iinfo(np.int8)
fits = (counts >= limits.min) & (counts <= limits.max)

print(limits.min, limits.max)  # -128、127。
print(fits.tolist())  # [True, True, False]。
print(counts.loc[~fits])  # 标签 2 的 128 超出范围。
if fits.all():
    narrowed = counts.astype("int8")
    print(narrowed.dtype)
else:
    print("保留 int64：输入包含超出 int8 范围的计数。")

-128 127
[True, True, False]
2    128
dtype: int64
保留 int64：输入包含超出 int8 范围的计数。


### 8.1 大整数不能先转浮点再求精确

to_numeric 的大整数处理不能只看是否报错，还要检查结果类型与精度。超出 64 位整数范围时，转换可能使用浮点表示并丢失整数精度。errors="coerce" 也不等于“逐位精确保留”。

下面用两个相邻的大整数文本观察边界。若它们只是编号，就保留文本；若要求精确整数运算，需要先选择支持相应范围的表示，不能靠事后 astype 恢复已丢失的数位。

In [14]:
texts = pd.Series(["18446744073709551616", "18446744073709551617"], dtype="str")
print(np.iinfo(np.int64).max, np.iinfo(np.uint64).max)
# 最大值分别为 9223372036854775807、18446744073709551615；两个输入均超过后者。
strict = pd.to_numeric(texts, errors="raise")
print(strict.tolist(), strict.dtype)
# 当前环境返回 object，保存两个不同的 Python 大整数，并没有得到 uint64 列。

floating = pd.to_numeric(texts, errors="coerce")
print(floating.tolist(), floating.dtype)  # 本例得到两个相同的 float64 近似值。
print(floating.iloc[0] == floating.iloc[1])  # True，原本不同的输入已无法区分。
print(int(floating.iloc[1]) == int(texts.iloc[1]))  # False：转回整数也不能恢复。
print(texts.tolist())  # 原始文本保留两个不同编号。

9223372036854775807

 18446744073709551615
[18446744073709551616, 18446744073709551617] object
[1.8446744073709552e+19, 1.8446744073709552e+19] float64
True
False
['18446744073709551616', '18446744073709551617']


## 9 自动整理可空类型

convert_dtypes 尝试把现有列转为合适的可空类型；默认后端为 numpy_nullable。整数值与缺失的组合可转为 Int64，实际包含小数的列可转为 Float64，布尔列可转为 boolean。

它根据现有数据判断，不理解业务含义，也不是数字文本解析器。下面先整理类型，再用 to_numeric 单独解析数量文本。

In [15]:
raw = pd.DataFrame({
    "count": [1.0, None, 3.0],
    "value": [1.5, None, 3.5],
    "valid": [True, None, False],
    "quantity_text": ["2", None, "4"],
})
converted = raw.convert_dtypes()

print(converted)
print(converted.dtypes)  # count、value、valid 分别为 Int64、Float64、boolean。
print(converted["quantity_text"].dtype)  # string：文本数字并未变成数值。
parsed = pd.to_numeric(converted["quantity_text"], errors="raise")
print(parsed)  # 2、<NA>、4，本例为 Int64。
print(raw.dtypes)  # 原表仍保留原先推断的 dtype。
print(converted.shape, converted.isna().sum().tolist())  # (3, 4)，每列缺失数都为 1。

   count  value  valid quantity_text
0      1    1.5   True             2
1   <NA>   <NA>   <NA>          <NA>
2      3    3.5  False             4
count              Int64
value            Float64
valid            boolean
quantity_text     string
dtype: object
string
0       2
1    <NA>
2       4
Name: quantity_text, dtype: Int64
count            float64
value            float64
valid             object
quantity_text        str
dtype: object
(3, 4) [1, 1, 1, 1]


## 10 按列类型选择

select_dtypes 按已经存在的 dtype 选列。include="number" 选择数值列；pandas 3 可用 include="str" 选择字符串专用列。只取 object 列时，用 include="object"、exclude="str" 明确排除字符串专用列。

当前版本的 include="object" 单独使用仍会兼容性地选中默认 str，并发出弃用警告；不要据此把 str 当作 object，也不要将它当成选择所有文本列的规则。

下面为数值处理准备子表，同时区分字符串列和混合对象列。这个操作不转换数据。

In [16]:
table = pd.DataFrame({
    "name": pd.Series(["甲", None], dtype="str"),
    "note": pd.Series(["正常", None], dtype="string"),
    "count": pd.Series([2, None], dtype="Int64"),
    "value": pd.Series([1.5, None], dtype="Float64"),
    "valid": pd.Series([True, None], dtype="boolean"),
    "mixed": pd.Series(["甲", 2], dtype="object"),
})
numeric = table.select_dtypes(include="number")
print(numeric)  # count、value 两列；boolean 不在这个数值子表中。
print(numeric.columns.tolist(), numeric.shape)  # ['count', 'value']，(2, 2)。
print(table.select_dtypes(include="str").columns.tolist())  # ['name', 'note']。
print(table.select_dtypes(include="object", exclude="str").columns.tolist())  # ['mixed']。
print(table.select_dtypes(include="number", exclude="floating").columns.tolist())
# 排除浮点列后，只剩 count。

   count  value
0      2    1.5
1   <NA>   <NA>

['count', 'value'] (2, 2)
['name', 'note']
['mixed']
['count']


## 11 array 与 to_numpy

Series.array 返回底层扩展数组，保留扩展类型；Series.to_numpy 返回 NumPy ndarray，类型与缺失表示可能改变。两者都只取值，不携带 Series 的行标签。

需要交给只接受 NumPy 数组的接口时，明确 dtype 和 na_value。下面将可空整数导出成 float64，用 NaN 表示缺失；本例整数较小，可以这样表示，不把它推广到所有大整数。

In [17]:
counts = pd.Series([2, None, 6], index=["A", "B", "C"], dtype="Int64")
extension = counts.array
values = counts.to_numpy(dtype="float64", na_value=np.nan)

print(type(extension).__name__, extension.dtype)  # IntegerArray，Int64。
print(extension)  # [2, <NA>, 6]。
print(type(values).__name__, values.dtype, values.shape)  # ndarray，float64，(3,)。
print(values)  # [2. nan 6.]。
print(counts.index.tolist())  # 标签 A、B、C 仍在原 Series 中。

IntegerArray

 Int64
<IntegerArray>
[2, <NA>, 6]
Length: 3, dtype: Int64
ndarray float64 (3,)
[ 2. nan  6.]
['A', 'B', 'C']


### 11.1 二维数组需要统一表示

DataFrame.to_numpy 把各列放入一个 NumPy 数组，需要为整个数组选取共同的 dtype。整数与浮点列可能统一为浮点；混合文本与数值通常需要 object。

转换可能发生复制。array 也不是独立副本；本章只观察返回内容，不通过这些对象修改原表。

In [18]:
numeric = pd.DataFrame({"count": [2, 4], "value": [1.5, 2.5]})
mixed = pd.DataFrame({"name": ["甲", "乙"], "count": [2, 4]})
numeric_values = numeric.to_numpy()
mixed_values = mixed.to_numpy()

print(numeric_values, numeric_values.dtype)  # 两列统一为 float64。
print(mixed_values, mixed_values.dtype)  # 数值和文本保存在 object 数组中。
print(numeric_values.shape, mixed_values.shape)  # 都为 (2, 2)，没有行列标签。

[[2.  1.5]
 [4.  2.5]] float64
[['甲' 2]
 ['乙' 4]] object
(2, 2) (2, 2)


## 12 选学：推断、降位宽与 Arrow

### 12.1 infer_objects 的软转换

infer_objects 只尝试改善 object 列，使用与构造 Series、DataFrame 时相同的推断规则。它不会把数字字符串按数值含义解析；需要可空目标时可考虑 convert_dtypes，需要解析文本时使用 to_numeric。

In [19]:
raw = pd.DataFrame({
    "objects": pd.Series([1, 2], dtype="object"),
    "text": pd.Series(["1", "2"], dtype="object"),
})
inferred = raw.infer_objects()

print(inferred)
print(inferred.dtypes)  # objects 为 int64；text 按 pandas 3 规则成为 str。
print(inferred["text"].tolist())  # ['1', '2']，仍是字符串。
print(raw.dtypes)  # 原表两列仍为 object。

   objects text
0        1    1
1        2    2
objects    int64
text         str
dtype: object
['1', '2']
objects    object
text       object
dtype: object


### 12.2 根据数据降位宽

to_numeric 的 downcast 可在成功解析后尝试较小类型。downcast="integer" 选择能容纳数据的较小有符号整数类型；它与预先强制指定 int8 不同，输入范围变大时结果也会变化。

降低浮点位宽还涉及精度取舍；这里只比较整数类型，不由更小的 dtype 推断整项任务的内存峰值或运行速度。

In [20]:
small = pd.to_numeric(pd.Series(["0", "127"]), downcast="integer")
larger = pd.to_numeric(pd.Series(["0", "128"]), downcast="integer")

print(small.tolist(), small.dtype)  # [0, 127]，int8。
print(larger.tolist(), larger.dtype)  # [0, 128]，int16；128 不能放进有符号 int8。
print(small.dtype.itemsize, larger.dtype.itemsize)  # 每个数值分别占 1、2 字节。

[0, 127] int8
[0, 128] int16
1 2


### 12.3 Arrow 后端入口

convert_dtypes(dtype_backend="pyarrow") 会尝试返回 ArrowDtype 列，使用已安装的 PyArrow。下面只比较相同数据的表示与缺失位置，不承诺它对所有操作更快。

默认 str 使用 Arrow 存储，不等于整张表已经采用 ArrowDtype；例如本章前面的 Int64 与这里的 int64[pyarrow] 就是不同的 dtype。

In [21]:
raw = pd.DataFrame({"count": [1, None, 3], "name": ["甲", None, "乙"]})
nullable = raw.convert_dtypes(dtype_backend="numpy_nullable")
arrow = raw.convert_dtypes(dtype_backend="pyarrow")

print(nullable.dtypes)  # Int64、string。
print(arrow.dtypes)  # int64[pyarrow]、string[pyarrow]。
print(arrow)
print(nullable.isna().equals(arrow.isna()))  # True：缺失位置相同。
print(arrow.shape, arrow.columns.tolist())  # (3, 2)，['count', 'name']。

count     Int64
name     string
dtype: object
count     int64[pyarrow]
name     string[pyarrow]
dtype: object
   count  name
0      1     甲
1   <NA>  <NA>
2      3     乙
True
(3, 2) ['count', 'name']


## 本章小结

（1）to_numeric 解析数值文本，astype 指定目标类型；保留原始列，才能区分解析失败和输入原有缺失。

（2）dtype 决定数值范围、缺失表示及运算行为。int64 与 Int64、float64 与 Float64、str 与 string 都需要区分。

（3）可空布尔按三值逻辑运算；未知不等于 False，但筛选时默认不选未知位置，业务需要保留时要明确规则。

（4）convert_dtypes 整理现有类型，select_dtypes 按已有类型选列。它们不替代业务校验或数字文本解析。

（5）array 保留扩展数组表示，to_numpy 面向 NumPy 接口；类型、缺失标记和标签都要重新核对，丢失的数值精度不能事后恢复。

## 练习

（1）解析数量文本并保留原始输入。把无效文本变为缺失，单独打印本次解析失败的记录；随后把结果转换为可空整数。不要把原有 None 当成新出现的解析失败。

In [22]:
raw = pd.DataFrame(
    {"quantity_text": ["4", "错误", None, "7"]}, index=["A", "B", "C", "D"]
)

# 在此解析、定位失败，并保存 Int64 新列。
# 检查：失败记录仅为 B；结果仍有 4 行，B、C 都缺失，A、D 为 4、7。
# 打印原输入、新结果、dtype 与缺失位置。

（2）先预测三种逻辑运算和两次筛选的结果，再运行核对。新增约束：所有未知判断都必须留下，交给人工复核。选择符合要求的筛选方法，说明保留未知与确认其为 True 的区别。

In [23]:
left = pd.Series([False, True, pd.NA], index=["A", "B", "C"], dtype="boolean")
right = pd.Series([pd.NA, pd.NA, False], index=left.index, dtype="boolean")
rows = pd.DataFrame({"count": [2, 4, 6]}, index=left.index)

print(left & right)
print(left | right)
print(left ^ right)
print(rows.loc[left])
print(rows.loc[left.fillna(True)])
# 在运行前写出预测；运行后检查结果标签和顺序，再解释新增约束下的选择。

A    False
B     <NA>
C    False
dtype: boolean
A    <NA>
B    True
C    <NA>
dtype: boolean
A    <NA>
B    <NA>
C    <NA>
dtype: boolean
   count
B      4
   count
B      4
C      6


（3）原任务只接收 0 到 100 的计数，后来输入增加了 128 和缺失值。说明为什么固定 int8 已不合适，选择能保留整数与缺失的目标类型；若业务仍限制 0 到 100，还应如何保留和报告违规记录？

In [24]:
counts = pd.Series([0, 100, 128, None], dtype="Int64")

# 在此查看目标整数类型的范围，选择类型并打印转换结果。
# 检查：128 不应绕回负数；缺失不能变成 0。
# 在注释中区分“类型能存储”与“符合业务范围”，并打印违规的非缺失记录。

（4）外部接口要求二维 float64 数组，缺失用 NaN，另需保留行列标签用于解释结果。完成转换并单独保存标签，解释为什么不能直接把 .array 当作整个 DataFrame 的导出方法。

In [25]:
table = pd.DataFrame({
    "count": pd.Series([2, None, 6], index=["A", "B", "C"], dtype="Int64"),
    "value": pd.Series([1.5, 2.5, None], index=["A", "B", "C"], dtype="Float64"),
})

# 在此保存行列标签，调用 to_numpy 并明确 dtype、na_value。
# 检查：数组为 (3, 2)、float64；两个缺失位置正确，标签顺序为 A、B、C。
# 在注释中解释 Series.array 与 DataFrame.to_numpy 的用途区别。

## 参考与引用来源

在线文档可能随发布更新；固定版本对照见下表 pandas v3.0.6 文档源码。API 参数与异常仍须结合所列页面的具体定位阅读。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| pandas 官方在线文档（课程基线 3.0.6） | [to_numeric](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html) 的 errors、downcast、dtype_backend 与大整数精度说明；[astype](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.astype.html) 的 dtype 映射、errors 与返回对象；[String dtype migration](https://pandas.pydata.org/docs/user_guide/migration-3-strings.html) 的 Background、Brief introduction、Missing value sentinel、setitem、astype(str) preserving missing values：默认 str、存储后端和缺失；[Working with text data](https://pandas.pydata.org/docs/user_guide/text.html) 的 Behavior differences、The four StringDtype variants：比较结果和缺失语义；[ExtensionDtype](https://pandas.pydata.org/docs/reference/api/pandas.api.extensions.ExtensionDtype.html) 的扩展类型基类（页面标注 3.0.5）；[StringDtype](https://pandas.pydata.org/docs/reference/api/pandas.StringDtype.html) 的 storage、na_value（页面标注 3.0.5）；[Nullable integer](https://pandas.pydata.org/docs/user_guide/integer_na.html) 的 Construction、Operations；[Float64Dtype](https://pandas.pydata.org/docs/reference/api/pandas.Float64Dtype.html)：可空浮点与 pd.NA；[Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html) 的 Values considered missing、NA semantics、NA in a boolean context：缺失检测与未知值；[Nullable Boolean](https://pandas.pydata.org/docs/user_guide/boolean.html) 的 Indexing with NA values、Kleene logical operations：三值真值表及筛选；[convert_dtypes](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.convert_dtypes.html) 的 Parameters、Notes：推断规则与后端；[select_dtypes](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.select_dtypes.html) 的 include、exclude、Notes，字符串选择按上述 3.0 迁移指南；[Series.array](https://pandas.pydata.org/docs/reference/api/pandas.Series.array.html) 的 Returns、Notes；[Series.to_numpy](https://pandas.pydata.org/docs/reference/api/pandas.Series.to_numpy.html) 的 dtype、na_value 与扩展类型转换；[DataFrame.to_numpy](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_numpy.html) 的共同类型与 Examples；[infer_objects](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.infer_objects.html) 的软转换规则；[PyArrow Functionality](https://pandas.pydata.org/docs/user_guide/pyarrow.html) 的 Data Structure Integration：ArrowDtype 与字符串存储的区别。 |
| NumPy 官方文档（2.5） | [iinfo](https://numpy.org/doc/2.5/reference/generated/numpy.iinfo.html) 的 min、max 与整数边界；[Data type objects](https://numpy.org/doc/2.5/reference/arrays.dtypes.html) 的 dtype 对象、object 类型与 itemsize。 |
| GitHub（pandas 官方源码） | [pandas/core/frame.py，v3.0.0](https://raw.githubusercontent.com/pandas-dev/pandas/v3.0.0/pandas/core/frame.py) 的 DataFrame.select_dtypes，5102–5152 行：默认 str 被 object 兼容选中的条件、显式 exclude 与弃用警告；同时核对本环境 3.0.6 官方发行包的同名方法实现。 |
| GitHub 官方项目（版本化来源） | pandas v3.0.6 文档源码：[migration-3-strings](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/migration-3-strings.rst)、[text](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/text.rst)、[integer_na](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/integer_na.rst)、[missing_data](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/missing_data.rst)、[boolean](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/boolean.rst)、[pyarrow](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/pyarrow.rst)；对应上列同名指南或发布说明的小节，作为固定版本对照。 |